# 🎓 Aulão Completo: O Guia Definitivo de `return`, `yield` e Mecanismos de Retorno em Python

Um guia prático e aprofundado cobrindo tudo o que você precisa saber sobre como funções e rotinas produzem e entregam dados em Python.

---

## 📑 Sumário
1. [O Clássico: return](#1.-O-Clássico:-return)
2. [O Gerador: yield](#2.-O-Gerador:-yield)
3. [A Delegação: yield from](#3.-A-Delegação:-yield-from)
4. [Comparativo de Memória e Performance: return vs yield](#4.-Comparativo-de-Memória-e-Performance:-return-vs-yield)
5. [Outras Formas de Retorno](#5.-Outras-Formas-de-Retorno)
   - 5.1 Lambdas e Generator Expressions
   - 5.2 Funções que Retornam Funções (Closures & Decorators)
   - 5.3 Context Managers (`__enter__` e `@contextmanager`)
   - 5.4 Assincronismo (`async def`, `await`, `async for` + `yield`)
6. [Tabela Resumo](#6.-Tabela-Resumo)

## 1. O Clássico: `return`

### 💡 Conceito
O `return` encerra **imediatamente** a execução da função e devolve um valor para quem a invocou. A pilha de chamadas (*call stack*) é desempilhada e as variáveis locais são descartadas da memória.

### 📌 Pontos Principais:
1. **Finalização imediata:** Qualquer código após o `return` é *unreachable* (inalcançável).
2. **Retorno implícito:** Sem uma cláusula `return` explícita, a função retorna `None`.
3. **Múltiplos valores:** Python empacota múltiplos valores separados por vírgula em uma **tupla**.
4. **Guard Clauses (Early Return):** Técnica para manter o código limpo, validando casos de erro primeiro e retornando logo.

In [ ]:
# 1.1 Comportamento básico e código inalcançável
def somar(a, b):
    resultado = a + b
    return resultado
    print('Isso NUNCA será impresso!')

print('Soma:', somar(10, 5))

# 1.2 Retorno implícito (None)
def log_mensagem(msg):
    print(f'[LOG]: {msg}')

res = log_mensagem('Processo iniciado')
print('Retorno de log_mensagem:', res)

# 1.3 Retornando múltiplos valores (Tupla)
def estatisticas(numeros):
    return min(numeros), max(numeros), sum(numeros) / len(numeros)

menor, maior, media = estatisticas([10, 20, 30, 40, 50])
print(f'Menor: {menor}, Maior: {maior}, Média: {media}')

# 1.4 Early Return (Guard Clauses)
def validar_usuario(user):
    if not user.get('ativo', False):
        return 'Erro: Usuário inativo'
    if user.get('idade', 0) < 18:
        return 'Erro: Usuário menor de idade'
    
    # Fluxo principal direto e sem aninhamento excessivo
    return f'Sucesso: Bem-vindo {user["nome"]}'

print(validar_usuario({'nome': 'Lucas', 'ativo': True, 'idade': 22}))
print(validar_usuario({'nome': 'Ana', 'ativo': False, 'idade': 25}))

## 2. O Gerador: `yield`

### 💡 Conceito
O `yield` transforma a função em um **Gerador (Generator)**.
- Ao invés de rodar até o final e destruir seu estado, a função **pausa** no `yield`, entrega o valor atual e **guarda todo o seu estado local na memória**.
- Quando solicitada novamente (via `next()` ou loop `for`), ela retoma **exatamente da linha onde pausou**.
- Ao terminar, levanta a exceção `StopIteration`.

In [ ]:
def contador_passo_a_passo():
    print('  [Passo 1] Iniciando função geradora...')
    yield 1
    print('  [Passo 2] Retomando após yield 1...')
    yield 2
    print('  [Passo 3] Retomando após yield 2...')
    yield 3
    print('  [Fim] Não há mais yields!')

# Criando o gerador (nenhum código dentro da função roda ainda!)
gen = contador_passo_a_passo()
print('Tipo:', type(gen))

print('\n--- Consumindo manualmente com next() ---')
print('Recebido:', next(gen))
print('Recebido:', next(gen))
print('Recebido:', next(gen))

try:
    next(gen)  # Vai lançar StopIteration
except StopIteration:
    print('StopIteration capturado: O gerador terminou!')

print('\n--- Consumindo com loop for ---')
for valor in contador_passo_a_passo():
    print('Loop recebeu:', valor)

## 3. A Delegação: `yield from`

### 💡 Conceito
Introduzido no Python 3.3, o `yield from` serve para **delegar** a produção de valores para outro iterável ou subgerador.

Ele elimina loops intermediários e estabelece um canal direto bidirecional entre o chamador e o subgerador, inclusive repassando o valor de `return` do subgerador.

In [ ]:
# 3.1 Delegando a múltiplos iteráveis
def mesclar_listas(lista_a, lista_b):
    yield from lista_a
    yield from lista_b

itens = list(mesclar_listas([1, 2, 3], ['A', 'B', 'C']))
print('Mesclado com yield from:', itens)

# 3.2 Capturando o valor de 'return' de um subgerador com yield from
def subgerador_calculo():
    yield 10
    yield 20
    return 'Resultado Final = 30'  # Em geradores, return define o valor da StopIteration

def gerador_principal():
    print('[Principal] Iniciando...')
    valor_retornado = yield from subgerador_calculo()
    print(f'[Principal] Subgerador retornou: {valor_retornado}')
    yield 999

for item in gerador_principal():
    print('Recebido no for:', item)

## 4. Comparativo de Memória e Performance: `return` vs `yield`

### 💡 Análise de Eficiência (Eager vs Lazy)
- **`return` (Eager Evaluation):** Gera toda a lista de uma só vez na memória RAM antes de entregar.
- **`yield` (Lazy Evaluation / Streaming):** Gera cada item sob demanda, mantendo uso de memória constante $O(1)$.

In [ ]:
import sys

N = 1_000_000

# Abordagem com return (Cria lista inteira na RAM)
def gerar_com_return(n):
    resultado = []
    for i in range(n):
        resultado.append(i * 2)
    return resultado

# Abordagem com yield (Gera sob demanda)
def gerar_com_yield(n):
    for i in range(n):
        yield i * 2

lista_completa = gerar_com_return(N)
gerador_lazy = gerar_com_yield(N)

tamanho_lista = sys.getsizeof(lista_completa)
tamanho_gerador = sys.getsizeof(gerador_lazy)

print(f'Tamanho da lista com return: {tamanho_lista:,} bytes (~{tamanho_lista / (1024*1024):.2f} MB)')
print(f'Tamanho do gerador com yield: {tamanho_gerador:,} bytes (~{tamanho_gerador / 1024:.2f} KB)')
print(f'Economia de memória: {tamanho_lista / tamanho_gerador:.0f}x menor!')

## 5. Outras Formas de Retorno em Python

Além do `return` e `yield` tradicionais, Python oferece diversos outros padrões e construções que produzem e retornam valores.

### 5.1 Lambdas e Generator Expressions
- **Lambda:** Função anônima com retorno implícito de uma expressão única.
- **List Comprehension:** Retorna uma lista concreta avaliada imediatamente.
- **Generator Expression:** Sintaxe com parênteses `()` que retorna um gerador lazy.

In [ ]:
# Lambda (retorno implícito)
quadrado = lambda x: x ** 2
print('Lambda quadrado(5):', quadrado(5))

# List Comprehension (Eager - Retorna lista)
pares_list = [x for x in range(10) if x % 2 == 0]
print('List comprehension:', pares_list)

# Generator Expression (Lazy - Retorna generator)
pares_gen = (x for x in range(10) if x % 2 == 0)
print('Generator expression:', pares_gen)
print('Primeiro valor:', next(pares_gen))
print('Segundo valor:', next(pares_gen))

### 5.2 Funções que Retornam Funções (Closures & Decorators)
Como funções em Python são objetos de primeira classe (*first-class citizens*), uma função pode retornar outra função, encapsulando dados do escopo externo (Closure).

In [ ]:
# Closure: Fábrica de funções
def criar_multiplicador(fator):
    def multiplicar(x):
        return x * fator
    return multiplicar  # Retorna o objeto função

dobrar = criar_multiplicador(2)
triplicar = criar_multiplicador(3)

print('dobrar(10):', dobrar(10))
print('triplicar(10):', triplicar(10))

# Decorator básico
def logger(func):
    def wrapper(*args, **kwargs):
        print(f'[LOG] Chamando {func.__name__}...')
        res = func(*args, **kwargs)
        print(f'[LOG] {func.__name__} finalizou com sucesso.')
        return res  # Retorna o resultado da função decorada
    return wrapper

@logger
def saudar(nome):
    return f'Olá, {nome}!'

print(saudar('Lucas'))

### 5.3 Context Managers (`with`, `__enter__` e `@contextmanager`)
No bloco `with objeto as variavel:`, a variável recebe o retorno do método `__enter__` ou o valor emitido pelo `yield` de um generator decorado com `@contextmanager`.

In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def cronometro(nome_etapa):
    print(f'⏱️ Iniciando: {nome_etapa}')
    inicio = time.perf_counter()
    try:
        # O valor no yield vai para o 'as variavel'
        yield f'Recurso alocado para {nome_etapa}'
    finally:
        fim = time.perf_counter()
        print(f'⏱️ {nome_etapa} concluído em {(fim - inicio)*1000:.3f} ms')

with cronometro('Cálculo de Potências') as info:
    print('Status recebido no with:', info)
    total = sum(i**2 for i in range(100_000))

### 5.4 Assincronismo: `async def`, `await` e Async Generators (`yield`)
- `async def` + `return`: Retorna uma **Coroutine** que é resolvida com `await`.
- `async def` + `yield`: Cria um **Async Generator**, consumido de forma não-bloqueante com `async for`.

In [ ]:
import asyncio

# 1. Coroutine com return
async def buscar_usuario_api(user_id):
    await asyncio.sleep(0.1)  # Simula I/O de rede assíncrono
    return {'id': user_id, 'nome': 'Lucas Heitor'}

# 2. Async Generator com yield
async def stream_eventos_assincrono(total=3):
    for i in range(1, total + 1):
        await asyncio.sleep(0.1)
        yield f'Evento #{i} recebido'

# Execução
async def main():
    print('--- Buscando usuário com await ---')
    user = await buscar_usuario_api(42)
    print('Retorno coroutine:', user)
    
    print('\n--- Consumindo stream assíncrono com async for ---')
    async for evento in stream_eventos_assincrono():
        print(evento)

# No Jupyter, podemos rodar diretamente await main()
await main()

## 6. Tabela Resumo

| Mecanismo | O que faz? | Mantém estado na memória? | Quando usar? |
| :--- | :--- | :---: | :--- |
| **`return`** | Encerra a função e devolve o resultado final | ❌ Não (desaloca a stack) | Quando o cálculo acabou e você quer o resultado pronto na hora |
| **`yield`** | Pausa a função, produz um item e aguarda o próximo `next()` | ✅ Sim | Streaming de dados, coleções grandes ou infinitas, economia de RAM |
| **`yield from`** | Delega a iteração/geração para outro iterável/gerador | ✅ Sim | Encadear geradores, modularizar sub-pipelines |
| **`lambda`** | Função rápida de linha única com retorno implícito | ❌ Não | Funções descartáveis (`sort(key=...)`, `map`, `filter`) |
| **`Closure`** | Função que gera e retorna outra função especializada | ✅ Sim (closure) | Fábricas de funções, Decorators, injeção de configurações |
| **`Async Generator`** | `yield` com suporte a `await` e I/O não-bloqueante | ✅ Sim | WebSockets, streaming de chunks de rede/HTTP, filas assíncronas |